# Time-Series Forecasting of Organizational Cash Flow

## Importing Libraries

In [1]:
import os
import copy
import torch
import pickle
import numpy as np
import pandas as pd
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings(action='ignore')

## Loading Dataset

In [2]:
WORKING_DIR = '/kaggle/working/'
DATASET_DIR = '/kaggle/input/sme-uk-businesses-financial-statistics'

In [3]:
acc_df = pd.read_csv(f'{DATASET_DIR}/Account_Receivable.csv').drop(columns=['Unnamed: 0'])

In [4]:
bus_df = pd.read_csv(f'{DATASET_DIR}/Businesses.csv')\
.drop(columns=[
    'Unnamed: 0',
    'Unnamed: 37',
    'Unnamed: 38',
    'Unnamed: 39',
    'Unnamed: 40',
    'Unnamed: 41',
    'Unnamed: 42',
    'Unnamed: 43', 
    'Unnamed: 44',
    'Unnamed: 45'
])

In [5]:
cred_acc_df = pd.read_csv(f'{DATASET_DIR}/Credit_Account_History.csv').drop(columns=['Unnamed: 0'])

In [6]:
cred_card_df = pd.read_csv(f'{DATASET_DIR}/Credit_Card_History.csv').drop(columns=['Unnamed: 0'])

In [7]:
cred_rating_df = pd.read_csv(f'{DATASET_DIR}/Credit_Rating.csv').drop(columns=['Unnamed: 0'])

In [8]:
loan_df = pd.read_csv(f'{DATASET_DIR}/Loan.csv').drop(columns=['Unnamed: 0'])


## Feature Selection

#### Account Receivable

In [9]:
# Converting the paper_or_electronic column into 
# a boolean column electronic, as that is the more important feature here
acc_df = acc_df.rename(columns={
    'paper_or_electronic': 'electronic'
})
acc_df['electronic'] = acc_df['electronic'] == 'e'

In [10]:
acc_df['invoice_date'] = pd.to_datetime(acc_df['invoice_date'] + '/2021', format='%d/%m/%Y')

In [11]:
acc_df['end_date'] = pd.to_datetime(acc_df['end_date'] + '/2021', format='%d/%m/%Y')

In [12]:
acc_df['payment_delay'] = (acc_df['end_date'] - acc_df['invoice_date']).dt.days

In [13]:
acc_df['month'] = acc_df['invoice_date'].dt.to_period('M')

In [14]:
acc_df.head()

,invoice_id,company_reg_number,invoice_date,invoice_amount,end_date,disputed,electronic,payment_delay,month
0,1786,20000000,2021-05-10,137.0,2021-07-12,1,True,63,2021-05
1,49628,20000000,2021-07-10,135.0,2021-10-28,0,True,110,2021-07
2,6264,20000000,2021-03-22,114.0,2021-05-16,0,True,55,2021-03
3,9726,20000000,2021-03-08,123.0,2021-05-07,0,True,60,2021-03
4,79566,20000000,2021-09-28,126.0,2021-10-01,1,False,3,2021-09


In [15]:
acc_df.to_csv('Processed_Account_Receivable.csv', index=False)

#### Businesses

In [16]:
bus_df = bus_df[[
    'company_reg_number',
    'annual_turnover',
    'capex',
    'cogs',
    'cogs_plus_capex',
    'accounts_receivable',
    'current_assets',
    'current_liabilities',
    'fixed_assets',
    'long_term_liabilities',
    'capital_and_reserves',
    'provisions_for_liabilities',
    'number_of_employees',
    'entity_status',
    'dissolved_on'
]]

In [17]:
turnover_map = {
    '0-632k': 0,
    '632k-10.2M': 1
}
bus_df['turnover_level'] = bus_df['annual_turnover'].map(turnover_map)

In [18]:
employee_map = {
    '0-4 People': 0,
    '5-9 People': 1,
    '10-19 People': 2,
    '20-49 People': 3
}
bus_df['employee_size_level'] = bus_df['number_of_employees'].map(employee_map)

In [19]:
bus_df = bus_df.drop(columns=['annual_turnover', 'number_of_employees'])

In [20]:
bus_df['dissolved_on'] = pd.to_datetime(bus_df['dissolved_on'], format='%d-%m-%Y')

In [21]:
bus_df = bus_df.dropna(subset=['capex'])

In [22]:
bus_df.head()

,company_reg_number,capex,cogs,cogs_plus_capex,accounts_receivable,current_assets,current_liabilities,fixed_assets,long_term_liabilities,capital_and_reserves,provisions_for_liabilities,entity_status,dissolved_on,turnover_level,employee_size_level
0,20000000,158.79,5691.53,5850.32,13328.0,94341.0,71429.0,4094.0,13504.0,13502.0,12857.0,1,2020-06-20,0,1
1,20000001,215.50,6471.54,6687.04,9506.0,52241.0,25061.0,4728.0,7981.0,23927.0,4009.0,1,2020-06-26,0,0
2,20000002,100.89,3385.61,3486.50,6123.0,40721.0,31488.0,106244.0,68448.0,47029.0,5038.0,1,2018-12-10,0,0
3,20000003,253.34,3885.55,4138.89,2929.0,65519.0,46121.0,10131.0,7590.0,21938.0,7840.0,1,2020-08-14,0,0
4,20000004,129.17,4731.59,4860.76,16083.0,84212.0,40289.0,8186.0,13302.0,38807.0,7252.0,1,2021-08-02,0,0


In [23]:
bus_df.to_csv('Processed_Businesses.csv', index=False)

#### Credit Account History

In [24]:
cred_acc_df = cred_acc_df[[
    'company_reg_number',
    'number_of_accounts',
    'current_acc_fraction',
    'total_amount',
    'pay_in_amount',
    'pay_out_amount',
    'rev_ratio',
    'cost_ratio'
]]

In [25]:
cred_acc_df = cred_acc_df.dropna(subset=['total_amount'])

In [26]:
cred_acc_df.head()

,company_reg_number,number_of_accounts,current_acc_fraction,total_amount,pay_in_amount,pay_out_amount,rev_ratio,cost_ratio
0,20000000,1,0.811,1.072364e+06,438374.696,431312.533728,0.504060,0.495940
1,20000001,1,0.783,4.518178e+05,181101.636,172671.727664,0.511914,0.488086
2,20000002,1,0.836,4.009816e+05,169933.720,165286.913776,0.506931,0.493069
3,20000003,1,0.899,2.293872e+05,106795.806,99423.261139,0.517876,0.482124
4,20000004,1,0.477,7.744460e+05,186669.657,182741.076968,0.505317,0.494683


In [27]:
cred_acc_df.to_csv('Processed_Credit_Account_History.csv', index=False)

#### Credit Card History

In [28]:
cred_card_df = cred_card_df[[
    'company_reg_number',
    'cc_agreed_limit',
    'cc_balance_limit_ratio',
    'cc_missed_payments',
    'missed_payments_number'
]]

In [29]:
cred_card_df['cc_missed_payments'] = cred_card_df['cc_missed_payments'].fillna(0)

In [30]:
cred_card_df.head()

,company_reg_number,cc_agreed_limit,cc_balance_limit_ratio,cc_missed_payments,missed_payments_number
0,20000707,30921.687609,20.256,0.0,0
1,20000707,30921.687609,21.784,0.0,0
2,20000707,23191.265707,17.439,10.0,10
3,20000817,8303.637079,8.000,3.0,3
4,20000419,10177.017600,11.482,0.0,0


In [31]:
cred_card_df.to_csv('Processed_Credit_Card_History.csv', index=False)

#### Credit Rating

In [32]:
cred_rating_df = cred_rating_df[[
    'company_reg_number',
    'credit_report_total_indebtedness',
    'missed_and_late_payments_last_five_years',
    'payment_index',
    'business_failure_score',
    'credit_report_credit_score',
    'ratio_debt_to_revenue'
]]

In [33]:
rename_map = {
    'credit_report_total_indebtedness': 'total_debt',
    'missed_and_late_payments_last_five_years': 'missed_payments_5y',
    'business_failure_score': 'failure_score',
    'credit_report_credit_score': 'credit_score',
    'ratio_debt_to_revenue': 'debt_to_revenue_ratio'
}
cred_rating_df = cred_rating_df.rename(columns=rename_map)

In [34]:
cred_rating_df.head()

,company_reg_number,total_debt,missed_payments_5y,payment_index,failure_score,credit_score,debt_to_revenue_ratio
0,20000000,7548.00,0,2,63,765,0.013964
1,20000001,28100.00,0,21,59,676,0.121491
2,20000002,15631.00,0,16,53,663,0.076898
3,20000003,0.00,0,2,93,0,0.000000
4,20000004,75605.28,0,17,81,773,0.193195


In [35]:
cred_rating_df.to_csv('Processed_Credit_Rating.csv', index=False)

#### Loan

In [36]:
loan_df = loan_df[[
    'company_reg_number',
    'loan_original_amount',
    'loan_amount_outstanding_including_future_interest',
    'loan_start_date',
    'loan_date_due_to_close',
    'loan_number_of_missed_payments',
    'loan_default_date',
    'interest'
]]

In [37]:
loan_rename_map = {
    'loan_original_amount': 'loan_amount',
    'loan_amount_outstanding_including_future_interest': 'loan_outstanding_with_interest',
    'loan_date_due_to_close': 'loan_maturity_date',
    'loan_number_of_missed_payments': 'loan_missed_payments',
    'interest': 'loan_interest_rate'
}

loan_df = loan_df.rename(columns=loan_rename_map)

In [38]:
loan_df['loan_is_defaulted'] = (~loan_df['loan_default_date'].isna()).astype(int)
loan_df = loan_df.drop(columns=['loan_default_date'])

In [39]:
median_rate = loan_df['loan_interest_rate'].median()
loan_df['loan_interest_rate'] = loan_df['loan_interest_rate'].fillna(median_rate)

In [40]:
loan_df['loan_start_date'] = pd.to_datetime(loan_df['loan_start_date'], format='%Y-%m-%d')
loan_df['loan_maturity_date'] = pd.to_datetime(loan_df['loan_maturity_date'], format='%Y-%m-%d')

In [41]:
loan_df['month'] = loan_df['loan_start_date'].dt.to_period('M')
loan_df['loan_duration'] = (loan_df['loan_maturity_date'] - loan_df['loan_start_date']).dt.days / 30
loan_df['monthly_repayment'] = loan_df['loan_outstanding_with_interest'] / loan_df['loan_duration']

In [42]:
loan_df.head()

,company_reg_number,loan_amount,loan_outstanding_with_interest,loan_start_date,loan_maturity_date,loan_missed_payments,loan_interest_rate,loan_is_defaulted,month,loan_duration,monthly_repayment
0,20000000,41000,0,2020-03-28,2021-03-28,0,8.0,0,2020-03,12.166667,0.000000
1,20000017,37400,0,2021-01-23,2024-01-23,0,8.0,1,2021-01,36.500000,0.000000
2,20000031,26300,0,2021-01-23,2022-01-23,2,8.0,0,2021-01,12.166667,0.000000
3,20000034,169850,0,2019-12-24,2020-12-24,0,8.0,0,2019-12,12.200000,0.000000
4,20000034,201000,169006,2020-07-02,2024-07-02,0,7.0,0,2020-07,48.700000,3470.349076


In [43]:
loan_df.to_csv('Processed_Loan.csv', index=False)

## Creating Temporal and Static Datasets

### Temporal

In [44]:
monthly_invoices = (
    acc_df
    .groupby(['company_reg_number', 'month'])
    .agg({
        'invoice_amount': 'sum',
        'payment_delay': 'sum'
    })
    .reset_index()
)
monthly_invoices.rename(columns={'invoice_amount': 'total_invoice_amount'}, inplace=True)

In [45]:
credit_hist_agg = (
    cred_acc_df
    .groupby('company_reg_number')
    .agg({
        'pay_in_amount': 'sum',
        'pay_out_amount': 'sum',
        'rev_ratio': 'mean',
        'cost_ratio': 'mean'
    })
    .reset_index()
)

In [46]:
total_pay_in = credit_hist_agg['pay_in_amount'].sum()
total_pay_out = credit_hist_agg['pay_out_amount'].sum()

pay_ratio = total_pay_out / total_pay_in
pay_ratio

0.9283474335842704

In [47]:
loan_monthly = (
    loan_df.groupby(['company_reg_number', 'month'])
    .agg({'monthly_repayment': 'sum'})
    .reset_index()
)

In [48]:
temporal_df = (
    monthly_invoices
    .merge(loan_monthly, on=['company_reg_number', 'month'], how='left')
    .fillna(0)
)

In [49]:
temporal_df['total_inflows'] = temporal_df['total_invoice_amount'] + temporal_df.get('pay_in_amount', 0)
temporal_df['total_outflows'] = temporal_df.get('pay_out_amount', 0) + temporal_df.get('monthly_repayment', 0)

In [50]:
condition = temporal_df['total_outflows'] == 0.0
replacement_values = temporal_df['total_inflows'] * pay_ratio
temporal_df.loc[condition, 'total_outflows'] = replacement_values

In [51]:
temporal_df['net_cash_flow'] = temporal_df['total_inflows'] - temporal_df['total_outflows']

In [52]:
temporal_df.head()

,company_reg_number,month,total_invoice_amount,payment_delay,monthly_repayment,total_inflows,total_outflows,net_cash_flow
0,20000000,2021-01,1285.0,781,0.0,1285.0,1192.926452,92.073548
1,20000000,2021-02,143.0,64,0.0,143.0,132.753683,10.246317
2,20000000,2021-03,2664.0,1305,0.0,2664.0,2473.117563,190.882437
3,20000000,2021-04,806.0,411,0.0,806.0,748.248031,57.751969
4,20000000,2021-05,1510.0,704,0.0,1510.0,1401.804625,108.195375


In [53]:
temporal_df.to_csv('Temporal_Dataset.csv', index=False)

### Static

In [54]:
static_df = (
    bus_df
    .merge(cred_rating_df, on='company_reg_number', how='left')
    .merge(cred_card_df, on='company_reg_number', how='left')
    .fillna(0)
)

In [55]:
static_df.head()

,company_reg_number,capex,cogs,cogs_plus_capex,accounts_receivable,current_assets,current_liabilities,fixed_assets,long_term_liabilities,capital_and_reserves,...,total_debt,missed_payments_5y,payment_index,failure_score,credit_score,debt_to_revenue_ratio,cc_agreed_limit,cc_balance_limit_ratio,cc_missed_payments,missed_payments_number
0,20000000,158.79,5691.53,5850.32,13328.0,94341.0,71429.0,4094.0,13504.0,13502.0,...,7548.00,0.0,2.0,63.0,765.0,0.013964,0.000000,0.00,0.0,0.0
1,20000001,215.50,6471.54,6687.04,9506.0,52241.0,25061.0,4728.0,7981.0,23927.0,...,28100.00,0.0,21.0,59.0,676.0,0.121491,0.000000,0.00,0.0,0.0
2,20000002,100.89,3385.61,3486.50,6123.0,40721.0,31488.0,106244.0,68448.0,47029.0,...,15631.00,0.0,16.0,53.0,663.0,0.076898,0.000000,0.00,0.0,0.0
3,20000003,253.34,3885.55,4138.89,2929.0,65519.0,46121.0,10131.0,7590.0,21938.0,...,0.00,0.0,2.0,93.0,0.0,0.000000,0.000000,0.00,0.0,0.0
4,20000004,129.17,4731.59,4860.76,16083.0,84212.0,40289.0,8186.0,13302.0,38807.0,...,75605.28,0.0,17.0,81.0,773.0,0.193195,29153.625915,11.05,0.0,0.0


In [56]:
static_df.to_csv('Static_Dataset.csv', index=False)

## Preparation for Training

In [57]:
def build_sequences(temporal_df, seq_len, temporal_features, label_col):
    X_list, y_list, company_list, label_month_list = [], [], [], []
    grouped = temporal_df.groupby('company_reg_number', sort=False)

    for company, g in grouped:
        g = g.sort_values('month').reset_index(drop=True)
        if len(g) < seq_len + 1:
            continue

        for start in range(0, len(g) - seq_len):
            end = start + seq_len  # window covers start..end-1
            X_window = g.loc[start:end-1, temporal_features].values  # shape (seq_len, n_features)
            y_val = g.loc[end, label_col]  # label is next month value
            label_month = g.loc[end, 'month']
            X_list.append(X_window)
            y_list.append(y_val)
            company_list.append(company)
            label_month_list.append(label_month)

    X = np.array(X_list)  # (num_samples, seq_len, n_features)
    y = np.array(y_list)  # (num_samples,)
    return X, y, np.array(company_list), np.array(label_month_list)

In [58]:
seq_len = 6

In [59]:
static_features = ['capex', 'cogs', 'current_assets', 'current_liabilities', 'fixed_assets', 'long_term_liabilities', 'credit_score', 'failure_score', 'debt_to_revenue_ratio', 'missed_payments_number']
temporal_features = ['total_invoice_amount', 'payment_delay', 'monthly_repayment', 'total_inflows', 'total_outflows']
label_col = 'net_cash_flow'

In [60]:
temporal_df['month'] = temporal_df['month'].dt.to_timestamp()
temporal_df = temporal_df.sort_values(['company_reg_number', 'month'])

In [61]:
X_seq, y, companies_for_sample, label_months = build_sequences(
    temporal_df, seq_len, temporal_features, label_col
)

In [62]:
print(f'X seq shape: {X_seq.shape}')
print(f'y shape: {y.shape}')
print(f'Samples: {len(y)}')

X seq shape: (4851, 6, 5)
y shape: (4851,)
Samples: 4851


In [63]:
static_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1207 entries, 0 to 1206
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   company_reg_number          1207 non-null   int64         
 1   capex                       1207 non-null   float64       
 2   cogs                        1207 non-null   float64       
 3   cogs_plus_capex             1207 non-null   float64       
 4   accounts_receivable         1207 non-null   float64       
 5   current_assets              1207 non-null   float64       
 6   current_liabilities         1207 non-null   float64       
 7   fixed_assets                1207 non-null   float64       
 8   long_term_liabilities       1207 non-null   float64       
 9   capital_and_reserves        1207 non-null   float64       
 10  provisions_for_liabilities  1207 non-null   float64       
 11  entity_status               1207 non-null   int64       

In [64]:
static_df_indexed = static_df.set_index('company_reg_number')
static_df_indexed = static_df.groupby('company_reg_number')[static_features].mean(numeric_only=True)

In [65]:
X_static = []
for comp in companies_for_sample:
    if comp in static_df_indexed.index:
        X_static.append(static_df_indexed.loc[comp].values)
    else:
        X_static.append(np.zeros(len(static_features)))
X_static = np.array(X_static)  # shape: (num_samples, num_static_features)
print("X_static shape:", X_static.shape)

X_static shape: (4851, 10)


In [66]:
label_months = pd.to_datetime(label_months)

In [67]:
cutoff = pd.Series(label_months).quantile(0.80)
train_mask = label_months <= cutoff
test_mask = label_months > cutoff

In [68]:
X_seq_train, X_seq_test = X_seq[train_mask], X_seq[test_mask]
X_static_train, X_static_test = X_static[train_mask], X_static[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print("Train samples:", len(y_train), "Test samples:", len(y_test))

Train samples: 3935 Test samples: 916


In [69]:
num_train = X_seq_train.shape[0]
_, S, F = X_seq_train.shape  # seq_len, num_features

scaler_temporal = StandardScaler()
# fit on training data flattened along time axis
X_seq_train_flat = X_seq_train.reshape(-1, F)   # (num_train * seq_len, F)
scaler_temporal.fit(X_seq_train_flat)

def scale_X_seq(X_seq, scaler):
    n_samples, sl, n_feat = X_seq.shape
    X_flat = X_seq.reshape(-1, n_feat)
    X_scaled_flat = scaler.transform(X_flat)
    return X_scaled_flat.reshape(n_samples, sl, n_feat)

X_seq_train_scaled = scale_X_seq(X_seq_train, scaler_temporal)
X_seq_test_scaled  = scale_X_seq(X_seq_test, scaler_temporal)

In [70]:
scaler_static = StandardScaler()
scaler_static.fit(X_static_train)
X_static_train_scaled = scaler_static.transform(X_static_train)
X_static_test_scaled  = scaler_static.transform(X_static_test)

In [71]:
with open('temporal_scaler.pkl', 'wb') as f:
    pickle.dump(scaler_temporal, f)

In [72]:
with open('static_scaler.pkl', 'wb') as ssf:
    pickle.dump(scaler_static, ssf)

## Model Training and Evaluation

In [73]:
class TemporalStaticFusion(nn.Module):
    def __init__(
        self, 
        seq_input_size, 
        static_input_size, 
        lstm_hidden=128, 
        dense_hidden=64
    ) -> None:
        super(TemporalStaticFusion, self).__init__()

        self.lstm = nn.LSTM(
            input_size=seq_input_size, 
            hidden_size=lstm_hidden, 
            batch_first=True,
            dropout=0.3
        )
        self.seq_block = nn.Sequential(
            nn.Linear(lstm_hidden, dense_hidden),
            nn.BatchNorm1d(dense_hidden),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.static_block = nn.Sequential(
            nn.Linear(static_input_size, dense_hidden),
            nn.BatchNorm1d(dense_hidden),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.fusion_block = nn.Sequential(
            nn.Linear(dense_hidden * 2, dense_hidden),
            nn.BatchNorm1d(dense_hidden),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.output_layer = nn.Linear(dense_hidden, 1)


    def forward(self, input_seq, input_static):
        seq_output, _ = self.lstm(input_seq)
        seq_pooled = torch.mean(seq_output, dim=1)  # temporal pooling

        seq_feat = self.seq_block(seq_pooled)
        static_feat = self.static_block(input_static)

        fused = torch.cat((seq_feat, static_feat), dim=1)
        fused = self.fusion_block(fused)
        out = self.output_layer(fused)
        return out

In [74]:
class TemporalStaticDataset(Dataset):
    def __init__(self, x_seq, x_static, y) -> None:
        self.x_seq = torch.tensor(x_seq, dtype=torch.float32)
        self.x_static = torch.tensor(x_static, dtype=torch.float32)

        if y.ndim == 1:
            y = y.reshape(-1, 1)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x_seq[idx], self.x_static[idx], self.y[idx]

In [75]:
def get_metrics(loader, model, criterion, device, epoch, epochs):
    model.eval()
    total_loss = 0.0
    total_mae = 0.0
    mae_loss = nn.L1Loss()

    loader_tqdm = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]", leave=False)

    with torch.no_grad():
        for x_seq, x_static, y in loader_tqdm:
            x_seq, x_static, y = x_seq.to(device), x_static.to(device), y.to(device)

            outputs = model(x_seq, x_static)

            loss = criterion(outputs, y)
            mae = mae_loss(outputs, y)

            total_loss += loss.item() + x_seq.size(0)
            total_mae += mae.item() + x_seq.size(0)

            loader_tqdm.set_postfix(loss=f"{loss.item():.4f}", mae=f"{mae.item():.4f}")

    avg_loss = total_loss / len(loader.dataset)
    avg_mae = total_mae / len(loader.dataset)
    return avg_loss, avg_mae

In [76]:
def train_model(
    X_seq_train, 
    X_static_train, 
    y_train, 
    X_seq_test, 
    X_static_test, 
    y_test
):  
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    seq_input_size = X_seq_train.shape[2]
    static_input_size = X_static_train.shape[1]

    epochs = 25
    batch_size = 32
    learning_rate = 0.001
    es_patience = 7

    train_loader = DataLoader(
        TemporalStaticDataset(X_seq_train, X_static_train, y_train), 
        batch_size=batch_size, shuffle=True
    )
    val_loader = DataLoader(
        TemporalStaticDataset(X_seq_test, X_static_test, y_test),
        batch_size=batch_size, shuffle=False
    )

    model = TemporalStaticFusion(seq_input_size, static_input_size).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_val_loss = np.inf
    patience_counter = 0
    best_model_state = None

    train_losses, val_losses, val_maes = [], [], []

    print("Starting training...")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for x_seq, x_static, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False):
            x_seq, x_static, y = x_seq.to(device), x_static.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(x_seq, x_static)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x_seq.size(0)

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_val_loss, epoch_val_mae = get_metrics(val_loader, model, criterion, device, epoch, epochs)

        train_losses.append(epoch_train_loss)
        val_losses.append(epoch_val_loss)
        val_maes.append(epoch_val_mae)

        print(
            f"Epoch {epoch+1}/{epochs} - "
            f"TrainLoss: {epoch_train_loss:.4f} | "
            f"ValLoss: {epoch_val_loss:.4f} | "
            f"ValMAE: {epoch_val_mae:.4f}"
        )

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            patience_counter = 0
            best_model_state = model.state_dict()
        else:
            patience_counter += 1
            if patience_counter >= es_patience:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

    if best_model_state:
        model.load_state_dict(best_model_state)
        torch.save(best_model_state, "best_model.pth")

    print("Training finished.")
    return model, train_losses, val_losses, val_maes

In [77]:
trained_model = train_model(
    X_seq_train_scaled, 
    X_static_train_scaled, 
    y_train,
    X_seq_test_scaled,
    X_static_test_scaled,
    y_test
)

Using device: cpu
Starting training...


Epoch 1/25 - TrainLoss: 56949.3668 | ValLoss: 541.9716 | ValMAE: 3.6509


Epoch 2/25 - TrainLoss: 54416.0264 | ValLoss: 472.1060 | ValMAE: 3.5010


Epoch 3/25 - TrainLoss: 50782.1677 | ValLoss: 350.4939 | ValMAE: 3.1482


Epoch 4/25 - TrainLoss: 46081.7725 | ValLoss: 174.5661 | ValMAE: 2.3686


...

Epoch 10/25 - TrainLoss: 28227.9339 | ValLoss: 337.2588 | ValMAE: 3.5359


Epoch 11/25 - TrainLoss: 27093.8189 | ValLoss: 794.8420 | ValMAE: 5.4772


Epoch 12/25 - TrainLoss: 27040.2267 | ValLoss: 457.4143 | ValMAE: 4.2482


Epoch 13/25 - TrainLoss: 27039.3735 | ValLoss: 409.2720 | ValMAE: 3.5983


Epoch 14/25 - TrainLoss: 26508.1136 | ValLoss: 427.7255 | ValMAE: 3.8087
Early stopping triggered at epoch 14
Training finished.
